<br>

# ANEEL

Para os pacotes que usam python, é necessário

In [ ]:
#!pip3 install arcgis

Definir a variável de ambiente `RESTAPI_USE_ARCPY` como `FALSE` é necessário para evitar que a biblioteca `restapi` mande mensagens de erro ou tente usar o `ArcPy`, que só está disponível para quem tem licença da ESRI.


In [ ]:
import os

os.environ['RESTAPI_USE_ARCPY'] = 'FALSE'

In [ ]:
import warnings

import requests
import restapi
from restapi import NAME, SERVICES, TYPE, ArcServer

import open_geodata as geo

In [ ]:
import json
import pprint
import tempfile
from pathlib import Path
# from arcgis.raster.functions import *
import geopandas as gpd

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
session = requests.Session()
client = restapi.RequestClient(session)
restapi.set_request_client(client)

In [ ]:
# connect to esri's sample server 6
url = 'https://sigel.aneel.gov.br/arcgis/rest/services'
url

In [ ]:
# Connect to restapi.ArcServer instance
ags = restapi.ArcServer(url)
ags

<br>

Com o uso do rest

In [ ]:
for x in ags.list_services():
    print(x, type(x))

In [ ]:
for root, services in ags.walk(ignore_folder_auth=True):
    print(f'Pasta: {root}')
    # print('\n'.join(f'- {item}' for item in services))
    for service in services:
        print(f'- {service}')

    print(f'-' * 60)

<br>

Obtem detalhes do serviço

In [ ]:
# Listas de Tipos de Serviço
ags.featureServices

In [ ]:
service = ags.getService(name_or_wildcard='SP')
service.name

In [ ]:
# Shapefile
print(service.name)
print(service.description)

# URL
print(service.url)

# Representação
print(repr(service))

# Formatos
print(service.supportedQueryFormats)

# Path
print(service.servicePath)

# documentInfo
print(service.documentInfo)

# Descrição
print(service.description)

# Informações do datum
print(service.initialExtent)
print(service.spatialReference)

In [ ]:
service.get_layer_url(name='ZEE')

In [ ]:
# Seleciona Layer no Serviço
lyr = service.layer(name_or_id=19)

In [ ]:
lyr_query = lyr.query(
    where='1=1',
    # Se exceed_limit=True, retorna todos os registros
    # Se exceed_limit=False, retorna apenas os primeiros 1000 registros
    exceed_limit=True,
    # ------------------------------------
    # Número de registros a serem retornados
    # Se records=None, retorna todos os registros
    # records=10,
    # ------------------------------------
    # Option to return a generator with a FeatureSet in chunks of each query group.
    # Use this to avoid memory errors when fetching many features. Defaults to False
    fetch_in_chunks=True,
)

lyr_query.geometryType

In [ ]:
# shp_file
temp_dir = tempfile.TemporaryDirectory()

In [ ]:
# Cria o caminho temporário em formato Path
temp_path = Path(temp_dir.name)
temp_path

In [ ]:
# ddd
restapi.exportFeatureSet(
    feature_set=lyr_query,
    #
    out_fc=str(temp_path / 'temp.shp'),
)

In [ ]:
gdf = gpd.read_file(filename=temp_path / 'temp.shp')

#
gdf.head()

<br>

-----

## ArcGIS


In [ ]:
from arcgis import geometry
from arcgis.geocoding import geocode
from arcgis.gis import GIS

In [ ]:
#url = "https://mapas.agenciapcj.org.br/arcgis/rest/services"

url = service.url
# Connect to the portal
gis = GIS(url)

In [ ]:
# Search for all feature services and feature collections in the portal
items = gis.content.search(query='type: "Feature Service" OR type: "Feature Collection"', max_items=5000)
items